# RouteAlert — เทรน Anti-Spoofing (Real Face vs Spoof Attempt)

เทรนจริงด้วย Transfer Learning จาก MobileNetV2 แล้ว export เป็น TFLite ให้ตรงกับ
`lib/core/ml/anti_spoofing_service.dart` ในแอป Flutter พอดี (โครงสร้างเดียวกับ
`train_accident_classifier_colab.ipynb`)

**ก่อนรัน:** เตรียมรูปภาพ 2 โฟลเดอร์ `real/` กับ `spoof/` บีบอัดเป็น `dataset.zip`

- `real/`: ภาพใบหน้าคนจริงถ่ายสดจากกล้อง มุม/แสง/คนหลากหลาย (ขอความยินยอมก่อนใช้ภาพใบหน้า
  ของเพื่อนร่วมทีม เพราะเป็นข้อมูลชีวมิติที่ละเอียดอ่อน)
- `spoof/`: ภาพการปลอมแปลง เช่น ถ่ายภาพที่พิมพ์ออกมา (printed photo), ถ่ายหน้าจอที่แสดง
  ใบหน้า (screen replay), ภาพจากวิดีโอเล่นซ้ำ

อย่างน้อยโฟลเดอร์ละ 80-150 รูป (ยิ่งหลากหลายสภาพแสง/มุม ยิ่งกันการปลอมแปลงได้ดีขึ้น)

In [ ]:
# ==== 1. เตรียมสภาพแวดล้อม ====
!pip install -q tensorflow

In [ ]:
# ==== 2. อัปโหลด dataset.zip ====
# โครงสร้างที่ต้องมีในไฟล์ zip:
#   dataset/
#     real/
#     spoof/
from google.colab import files
uploaded = files.upload()  # เลือก dataset.zip ที่เตรียมไว้

import zipfile
with zipfile.ZipFile('dataset.zip', 'r') as zip_ref:
    zip_ref.extractall('dataset')

In [ ]:
# ==== 3. โหลด dataset ====
import tensorflow as tf

IMG_SIZE = 224
BATCH_SIZE = 16

train_ds = tf.keras.utils.image_dataset_from_directory(
    'dataset',
    validation_split=0.2,
    subset='training',
    seed=42,
    image_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    'dataset',
    validation_split=0.2,
    subset='validation',
    seed=42,
    image_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
)

class_names = train_ds.class_names
print('Classes (เรียงตามตัวอักษร):', class_names)
# ต้องได้ ['real', 'spoof'] — ลำดับนี้จะตรงกับ anti_spoofing_labels.txt ที่ export ออกไป
# (โค้ด Dart หา label ที่มีคำว่า 'real'/'live'/'จริง' เป็นตัวชี้ว่า "ผ่านการตรวจสอบ")

data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip('horizontal'),
    tf.keras.layers.RandomRotation(0.05),
    tf.keras.layers.RandomBrightness(0.15),
    tf.keras.layers.RandomContrast(0.15),
])

AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.cache().shuffle(200).prefetch(buffer_size=AUTOTUNE)
val_ds = val_ds.cache().prefetch(buffer_size=AUTOTUNE)

In [ ]:
# ==== 4. สร้างโมเดล (Transfer Learning จาก MobileNetV2) ====
preprocess_input = tf.keras.applications.mobilenet_v2.preprocess_input

base_model = tf.keras.applications.MobileNetV2(
    input_shape=(IMG_SIZE, IMG_SIZE, 3),
    include_top=False,
    weights='imagenet',
)
base_model.trainable = False

inputs = tf.keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
x = data_augmentation(inputs)
x = preprocess_input(x)
x = base_model(x, training=False)
x = tf.keras.layers.GlobalAveragePooling2D()(x)
x = tf.keras.layers.Dropout(0.3)(x)
outputs = tf.keras.layers.Dense(len(class_names), activation='softmax')(x)
model = tf.keras.Model(inputs, outputs)

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.0005),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy'],
)
model.summary()

In [ ]:
# ==== 5. เทรน ====
EPOCHS = 15
history = model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS)

In [ ]:
# ==== 6. ตรวจผลลัพธ์ก่อน export ====
# สำหรับ anti-spoofing ควรได้ validation accuracy สูง (>90%) ก่อนนำไปใช้จริง
# เพราะเกี่ยวกับความปลอดภัยของระบบ login — ถ้าต่ำกว่านี้ให้เก็บข้อมูลเพิ่ม
loss, acc = model.evaluate(val_ds)
print(f'Validation accuracy: {acc*100:.1f}%')

In [ ]:
# ==== 7. Export เป็น TFLite (ให้ตรงกับที่ anti_spoofing_service.dart คาดไว้) ====
converter = tf.lite.TFLiteConverter.from_keras_model(model)
tflite_model = converter.convert()

with open('anti_spoofing.tflite', 'wb') as f:
    f.write(tflite_model)

with open('anti_spoofing_labels.txt', 'w') as f:
    for name in class_names:
        f.write(name + '\n')

print('Export เสร็จแล้ว: anti_spoofing.tflite, anti_spoofing_labels.txt')

In [ ]:
# ==== 8. ดาวน์โหลดไฟล์ ====
# นำไฟล์ทั้งสองไปวางที่ assets/models/ ในโปรเจกต์ Flutter แล้วรันแอปได้เลย
files.download('anti_spoofing.tflite')
files.download('anti_spoofing_labels.txt')